[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/24_Change_Management.ipynb)

# 变更管理 —— 实用指南

Semantica 的 `change_management` 模块为知识图谱和本体提供版本管理、审计轨迹和数据完整性检查:

- **`ChangeLogEntry`** —— 标准化的变更元数据(经过校验的时间戳/作者)
- **`InMemoryVersionStorage` / `SQLiteVersionStorage`** —— 支持命名标签的版本快照存储
- **`compute_checksum` / `verify_checksum`** —— SHA-256 完整性校验

本 Notebook 运行一个完整的 保存 → 打标签 → 校验 → 篡改检测 的循环。所有输出都是真实执行的结果,并在撰写时对照仓库中的 `semantica/change_management/` 源码验证过(`pip install` 单元格可能会获取到行为略有不同的较新版本)。

In [1]:
!pip install -q semantica

## 1)`ChangeLogEntry` 记录*谁*在*何时*修改了*什么*

`author` 必须是有效的电子邮件 —— 数据类在构造时会进行校验(否则抛出 `ValidationError`),从而保持审计轨迹的整洁。

In [2]:
from semantica.change_management import ChangeLogEntry

entry = ChangeLogEntry(
    timestamp="2026-08-15T09:00:00Z",
    author="demo@example.com",
    description="initial version",
)
entry

ChangeLogEntry(timestamp='2026-08-15T09:00:00Z', author='demo@example.com', description='initial version', change_id=None, related_changes=[])

## 2)保存带版本的快照

快照是一个字典,必须包含 `label`,再加上你的载荷。这里我们附上知识图谱数据、变更日志,以及一个 SHA-256 `checksum`,它是对除 checksum 字段本身之外的所有内容计算得出的。

In [3]:
from semantica.change_management import InMemoryVersionStorage, compute_checksum

storage = InMemoryVersionStorage()

snapshot = {
    "label": "v1.0.0",
    "data": {"entities": {"acme": {"type": "Company"}}},
    "change_log": {
        "timestamp": entry.timestamp,
        "author": entry.author,
        "description": entry.description,
    },
}
snapshot["checksum"] = compute_checksum({k: v for k, v in snapshot.items() if k != "checksum"})

storage.save(snapshot)
storage.exists("v1.0.0")

True

## 3)命名标签可为发布固定版本

`save_tag` / `get_tag` 将稳定的名称(例如 `release`)映射到版本标签,使使用者不受标签变动的影响。

In [4]:
storage.save_tag("release", "v1.0.0")

storage.get_tag("release"), [s["label"] for s in storage.list_all()]

('v1.0.0', ['v1.0.0'])

## 4)验证完整性 —— 并发现篡改

`verify_checksum(snapshot)` 会对快照(去掉其 `checksum` 字段)重新计算 SHA-256 并进行比较。数据中哪怕只有一个字符被改动,结果也会变为 `False`。

In [5]:
from semantica.change_management import verify_checksum

stored = storage.get("v1.0.0")
print("intact:", verify_checksum(stored))

tampered = storage.get("v1.0.0")
tampered["data"]["entities"]["acme"]["note"] = "mutated after the fact"
print("tampered:", verify_checksum(tampered))

intact: True
tampered: False


## 5)淘汰一个版本

`delete(label)` 会删除一个快照;指向它的标签需要由你自己负责更新。

In [6]:
storage.delete("v1.0.0")
storage.exists("v1.0.0")

False

## 总结

| 任务 | API |
|---|---|
| 记录审计元数据 | `ChangeLogEntry(timestamp, author=email, description)` |
| 持久化一个版本 | `InMemoryVersionStorage().save({"label": ..., ...})` |
| 固定一个发布名称 | `save_tag("release", "v1.0.0")` / `get_tag("release")` |
| 完整性检查 | `compute_checksum(snap)` / `verify_checksum(snap)` |
| 持久化后端 | `SQLiteVersionStorage(path)` —— 接口相同 |

另请参阅 `semantica/change_management/change_management_usage.md`,了解管理器类(`TemporalVersionManager`、`OntologyVersionManager`)。